In [ ]:
# auto_champions.ipynb -- the overnight CONDUCTOR. Run on exactly ONE pod
# (others run auto_stop_fulln.ipynb). Fully autonomous chain:
#   1. WATCH   : wait until every FULL-N combo is terminal (stall guard too);
#   2. KILL    : stop local training (supervisor SIGTERM -> lease self-revokes);
#   3. BATTERY : warmup once, then one eval_battery_worker per GPU -> full
#                downstream metrics for every done full-n combo (streams
#                grid_metrics.json);
#   4. SELECT  : per-metric top-K intersection -> champions.json;
#   5. PRUNE   : soft-mark every non-champion-family small-n combo failed;
#   6. RESUME  : relaunch the training supervisor here -> champion ladders;
#   7. FINISH  : when the whole (pruned) grid is terminal, stop training and
#                self-stop this pod via the hardened ladder.
import json, os, re, signal, socket, subprocess, sys, time
from pathlib import Path

RUNPOD_API_KEY_OVERRIDE = ""
REPO = "/workspace/stable-query-latent"
OUT_DIR = "VICReg_review/heads/cloud_full_sweep_a100"
SWEEP_YAML = "VICReg_review/sweep/sweep.yaml"
LOG_DIR = "/workspace/stable_query_latent_logs"
POLL_SECONDS = 120
STALL_HOURS = 4
TOP_K = 50               # champion selection cut (intersection <= TOP_K)
LADDER_NS = (1000, 500, 200)   # scaling points kept for champion families;
                               # every other small n (e.g. 1500) is pruned too
                               # -- three points draw the curve, 1500 is too
                               # close to 2000 to pay for
AUTO_PRUNE = True        # False = select champions but keep the whole tail
RESUME_TRAINING = True   # False = stop after champions.json (no tail training)
STOP_POD_WHEN_ALL_DONE = True
CLAIM_TTL = 7200

os.makedirs(LOG_DIR, exist_ok=True)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
from VICReg_review import pod_selfstop
from VICReg_review.sweep.config import SweepConfig

pod_id, api_key, ctl = pod_selfstop.preflight(RUNPOD_API_KEY_OVERRIDE)
root = Path(REPO) / OUT_DIR
HOST = socket.gethostname()
_m = re.fullmatch(r'Pod_(\d+)', Path.cwd().name)
VM_ID = f'VM{_m.group(1)}' if _m else HOST

cfg = SweepConfig.load(f'{REPO}/{SWEEP_YAML}')
combos = list(cfg.iter_combos())
_counts = [int(c.train_games) for c in combos]
FULL_N = 0 if any(n <= 0 for n in _counts) else max(_counts)
fulln_ids = [c.combo_id for c in combos if int(c.train_games) == FULL_N]
print(f'conductor: id={VM_ID} host={HOST}; {len(fulln_ids)} FULL-N combos; '
      f'TOP_K={TOP_K} prune={AUTO_PRUNE} resume={RESUME_TRAINING}')


def _read(p):
    try:
        return json.loads(Path(p).read_text(encoding='utf-8'))
    except Exception:
        return None


def _terminal(cid):
    d = root / cid
    if (d / 'done.json').exists() or (d / 'failed.json').exists():
        return True
    man = _read(d / 'vicreg_review_h5_manifest.json')
    return bool(man) and man.get('status') == 'done'


def kill_local_training(label):
    print(f'conductor: {label} -- stopping local training', flush=True)
    subprocess.run(['pkill', '-f', 'sweep/supervisor.py'])
    time.sleep(10)
    subprocess.run(['pkill', '-9', '-f', 'sweep/worker.py'])


def gpus():
    try:
        out = subprocess.run(['nvidia-smi', '--query-gpu=index', '--format=csv,noheader'],
                             capture_output=True, text=True, timeout=5).stdout
        return [l.strip() for l in out.splitlines() if l.strip()] or ['0']
    except Exception:
        return ['0']


def watch(ids, label, stall_hours=STALL_HOURS):
    """Poll until every id is terminal. False on stall."""
    last_n, last_ts = None, time.time()
    while True:
        left = [c for c in ids if not _terminal(c)]
        n = len(left)
        if last_n is None or n < last_n:
            last_n, last_ts = n, time.time()
        if n == 0:
            print(f'conductor: {label} complete', flush=True)
            return True
        if time.time() - last_ts > stall_hours * 3600:
            print(f'conductor: STALL in {label}: {n} remaining, no progress for '
                  f'{stall_hours}h', flush=True)
            return False
        print(f'{time.strftime("%H:%M:%S")} {label}: {n} remaining '
              f'(no-progress={((time.time() - last_ts) / 60):.0f}m)', flush=True)
        time.sleep(POLL_SECONDS)


def run_battery():
    warm = subprocess.run(
        [sys.executable, '-u', 'VICReg_review/eval_battery_worker.py', '--warmup-only',
         '--out-dir', OUT_DIR, '--sweep-yaml', SWEEP_YAML, '--worker-id', VM_ID,
         '--logout-address', f'{LOG_DIR}/auto_champ_warmup.log'], cwd=REPO)
    if warm.returncode != 0:
        print(f'!! warmup exited rc={warm.returncode}; continuing (workers rebuild '
              f'caches on demand)', flush=True)
    procs = []
    for g in gpus():
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))
        log = f'{LOG_DIR}/auto_champ_gpu{g}.log'
        p = subprocess.Popen(
            [sys.executable, '-u', 'VICReg_review/eval_battery_worker.py',
             '--out-dir', OUT_DIR, '--sweep-yaml', SWEEP_YAML,
             '--claim-ttl', str(CLAIM_TTL), '--worker-id', VM_ID,
             '--logout-address', log], env=env, cwd=REPO)
        procs.append((g, p, log))
        print(f'battery worker gpu{g} pid={p.pid} -> {log}', flush=True)
    for g, p, log in procs:
        rc = p.wait()
        tail = ''
        try:
            tail = Path(log).read_text(encoding='utf-8', errors='replace').splitlines()[-1]
        except Exception:
            pass
        print(f'battery worker gpu{g} exit={rc}  last: {tail}', flush=True)


def select_champions():
    mp = root / 'grid_metrics.json'
    metrics = json.loads(mp.read_text(encoding='utf-8'))
    rows = [dict(r) for r in metrics['rows']]
    METRICS = {'tag_f1': ('tag_f1', True), 'identity': ('identity_hit_at_1', True),
               'variant_drop': ('variant_drop_mean', False),
               'selectivity': ('selectivity', True)}
    import math

    def ok(v):
        return v is not None and not (isinstance(v, float) and math.isnan(v))

    top_sets, rank_maps = {}, {}
    for mname, (key, hi) in METRICS.items():
        scored = sorted((r for r in rows if ok(r.get(key))),
                        key=lambda r: r[key], reverse=hi)
        rank_maps[mname] = {r['combo_id']: i + 1 for i, r in enumerate(scored)}
        top_sets[mname] = [r['combo_id'] for r in scored[:TOP_K]]
    ids = set(r['combo_id'] for r in rows)
    for s_ in top_sets.values():
        ids &= set(s_)
    for r in rows:
        r['ranks'] = {m: rank_maps[m].get(r['combo_id']) for m in METRICS}
        r['rank_sum'] = sum(v for v in r['ranks'].values() if v is not None)
    champions = sorted((r for r in rows if r['combo_id'] in ids),
                       key=lambda r: r['rank_sum'])
    payload = {'created_at': time.strftime('%Y-%m-%dT%H:%M:%S'),
               'source_metrics': {'path': str(mp), 'created_at': metrics.get('created_at'),
                                  'git_commit': metrics.get('git_commit'),
                                  'pool_size': metrics.get('pool_size')},
               'full_n': metrics.get('full_n'), 'top_k': TOP_K,
               'selection': 'auto_champions: intersection of per-metric top-K',
               'metrics': {m: {'key': k, 'higher_is_better': hi}
                           for m, (k, hi) in METRICS.items()},
               'top_sets': top_sets, 'champions': champions}
    out = root / 'champions.json'
    tmp = out.with_name(f'{out.name}.tmp.{HOST}.{os.getpid()}')
    tmp.write_text(json.dumps(payload, ensure_ascii=False, indent=1), encoding='utf-8')
    tmp.replace(out)
    print(f'conductor: {len(champions)} champions (pool {len(rows)}, '
          f'missing {len(metrics.get("missing_reports", []))}) -> {out}', flush=True)
    return champions


def prune_tail(champions):
    families = {(r['output_dim'], r['num_latents'], r['view'], r['arm'])
                for r in champions}
    pruned = kept = 0
    ladder = set(int(n) for n in LADDER_NS)
    for c in combos:
        if int(c.train_games) == FULL_N:
            continue
        if int(c.train_games) in ladder and \
                (c.output_dim, c.num_latents, c.view, c.arm) in families:
            kept += 1
            continue
        d = root / c.combo_id
        if _terminal(c.combo_id):
            continue
        f = d / 'failed.json'
        f.parent.mkdir(parents=True, exist_ok=True)
        try:
            fd = os.open(str(f), os.O_CREAT | os.O_EXCL | os.O_WRONLY, 0o644)
        except OSError:
            continue
        try:
            os.write(fd, json.dumps({
                'vm': f'auto_champions@{HOST}',
                'error': 'pruned: non-champion family (soft; delete this file to '
                         're-enable)',
                'ts': time.time()}, ensure_ascii=False, indent=2).encode('utf-8'))
        finally:
            os.close(fd)
        pruned += 1
    print(f'conductor: pruned {pruned} small-n combos (non-champion families + off-ladder n); '
          f'{kept} champion-ladder combos kept', flush=True)


def resume_training():
    work = os.path.join('/tmp', 'stable_query_latent', 'work', VM_ID)
    os.makedirs(work, exist_ok=True)
    cmd = [sys.executable, '-u', 'VICReg_review/sweep/supervisor.py',
           '--config', SWEEP_YAML,
           '--h5', f'{REPO}/game_review_data/embedding_h5.h5',
           '--gpus', ','.join(gpus()),
           '--out-dir', OUT_DIR, '--work-dir', work,
           '--local-data-dir', '/root/data',
           '--vm-name', VM_ID,
           '--logout-address', f'{LOG_DIR}/pipeline_{VM_ID}.log']
    print('conductor: resuming training:', ' '.join(cmd), flush=True)
    return subprocess.Popen(cmd, cwd=REPO)


# ------------------------------- the chain -----------------------------------
ok = watch(fulln_ids, 'FULL-N')
kill_local_training('FULL-N phase over' if ok else 'stall abort')
if not ok:
    if STOP_POD_WHEN_ALL_DONE:
        pod_selfstop.stop_pod(pod_id, api_key, ctl)
else:
    run_battery()
    champions = select_champions()
    if AUTO_PRUNE and champions:
        prune_tail(champions)
    elif AUTO_PRUNE:
        print('!! zero champions (empty intersection) -- SKIPPING prune; the whole '
              'tail stays enabled. Lower TOP_K by hand tomorrow.', flush=True)
    if RESUME_TRAINING:
        proc = resume_training()
        all_ids = [c.combo_id for c in combos]
        finished = watch(all_ids, 'champion tail (whole grid)')
        kill_local_training('grid complete' if finished else 'tail stall abort')
        try:
            proc.wait(timeout=30)
        except Exception:
            pass
    if STOP_POD_WHEN_ALL_DONE:
        pod_selfstop.stop_pod(pod_id, api_key, ctl)
print('conductor: chain finished.', flush=True)
